# Notebook 03: YOLO11s-P2 + Mish

**Model**: YOLO11s-P2 with Mish activation in selected neck blocks

**Architectural Change**: Replace SiLU with Mish in neck/head C3k2 blocks

**Mish**: f(x) = x * tanh(softplus(x)) - smoother gradient flow than SiLU

In [1]:
import sys
import torch
import platform

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DEVICE = 0
    print(f"Current device: cuda:0")
else:
    print("CUDA NOT AVAILABLE - Training will proceed on CPU")
    print(f"Reason: PyTorch build = {torch.__version__} (CPU-only build)")
    DEVICE = "cpu"
    print(f"Current device: cpu")
print(f"OS: {platform.system()} {platform.release()}")
print("=" * 60)


ENVIRONMENT CHECK
Python version: 3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]
PyTorch version: 2.11.0+cu128
Ultralytics version: 8.4.92


CUDA available: True
CUDA version: 12.8
GPU name: NVIDIA GeForce RTX 3060 Laptop GPU
GPU memory: 6.00 GB
Current device: cuda:0
OS: Windows 11


In [2]:
# Shared training configuration - MUST be identical for all 5 models
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_CONFIG = {
    "data": r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\dataset\data.yaml",
    "imgsz": 640,
    "epochs": 100,
    "patience": 20,
    "batch": 8,
    "device": "cuda",
    "workers": 0,  # Windows safety
    "seed": SEED,
    "deterministic": True,
    "amp": True,  # AMP only with CUDA
    "pretrained": True,
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "verbose": True,
}

RESULTS_DIR = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Training configuration loaded:")
for k, v in TRAIN_CONFIG.items():
    if k != "data":
        print(f"  {k}: {v}")


Training configuration loaded:
  imgsz: 640
  epochs: 100
  patience: 20
  batch: 8
  device: cuda
  workers: 0
  seed: 42
  deterministic: True
  amp: True
  pretrained: True
  optimizer: auto
  lr0: 0.01
  lrf: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3.0
  warmup_momentum: 0.8
  warmup_bias_lr: 0.1
  box: 7.5
  cls: 0.5
  dfl: 1.5
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  degrees: 0.0
  translate: 0.1
  scale: 0.5
  shear: 0.0
  perspective: 0.0
  flipud: 0.0
  fliplr: 0.5
  mosaic: 1.0
  mixup: 0.0
  copy_paste: 0.0
  verbose: True


In [3]:
def benchmark_model(model_path, device, imgsz=640, warmup=20, runs=100):
    """Controlled latency benchmark for a YOLO model."""
    import time
    from ultralytics import YOLO
    
    model = YOLO(model_path)
    
    # Create dummy input
    dummy = torch.randn(1, 3, imgsz, imgsz)
    if device != "cpu":
        dummy = dummy.to(f"cuda:{device}")
    
    # Warmup
    print(f"Warming up ({warmup} iterations)...")
    for _ in range(warmup):
        _ = model.predict(source=dummy, verbose=False, device=device)
    
    # Timed runs
    print(f"Benchmarking ({runs} iterations)...")
    latencies = []
    for _ in range(runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = model.predict(source=dummy, verbose=False, device=device)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)  # ms
    
    mean_lat = np.mean(latencies)
    std_lat = np.std(latencies)
    fps = 1000.0 / mean_lat
    
    print(f"Latency: {mean_lat:.2f} +/- {std_lat:.2f} ms")
    print(f"FPS: {fps:.1f}")
    
    del model
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return mean_lat, std_lat, fps


In [4]:
def save_experiment_results(model_name, results, model_path, benchmark_results, results_dir):
    """Save experiment results to JSON for comparison notebook."""
    import os, json
    
    mean_lat, std_lat, fps = benchmark_results
    
    # Get model file size
    model_size_bytes = os.path.getsize(model_path)
    model_size_mb = model_size_bytes / (1024 * 1024)
    
    # Get model info
    from ultralytics import YOLO
    model = YOLO(model_path)
    info = model.info()
    if isinstance(info, tuple) and len(info) >= 4:
        n_layers, n_params, n_grads, gflops = info[:4]
    else:
        n_layers = len(list(model.model.modules()))
        n_params = sum(p.numel() for p in model.model.parameters())
        n_grads = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
        gflops = 0.0
    
    # Extract metrics from results
    metrics = {}
    if hasattr(results, 'results_dict'):
        rd = results.results_dict
        metrics["precision"] = rd.get("metrics/precision(B)", 0)
        metrics["recall"] = rd.get("metrics/recall(B)", 0)
        metrics["mAP50"] = rd.get("metrics/mAP50(B)", 0)
        metrics["mAP50-95"] = rd.get("metrics/mAP50-95(B)", 0)
    
    # Per-class metrics if available
    per_class = {}
    if hasattr(results, 'box'):
        box = results.box
        if hasattr(box, 'ap50') and box.ap50 is not None:
            class_names = {0: "Pothole", 1: "Crack", 2: "Manhole"}
            for i, name in class_names.items():
                if i < len(box.ap50):
                    per_class[name] = {
                        "AP50": float(box.ap50[i]),
                        "AP50-95": float(box.ap[i]) if hasattr(box, 'ap') and i < len(box.ap) else 0,
                        "precision": float(box.p[i]) if hasattr(box, 'p') and i < len(box.p) else 0,
                        "recall": float(box.r[i]) if hasattr(box, 'r') and i < len(box.r) else 0,
                    }
    
    result_data = {
        "model_name": model_name,
        "model_path": model_path,
        "n_layers": int(n_layers),
        "n_params": int(n_params),
        "n_grads": int(n_grads),
        "gflops": float(gflops),
        "model_size_mb": float(model_size_mb),
        "precision": float(metrics.get("precision", 0)),
        "recall": float(metrics.get("recall", 0)),
        "mAP50": float(metrics.get("mAP50", 0)),
        "mAP50-95": float(metrics.get("mAP50-95", 0)),
        "latency_ms": float(mean_lat),
        "latency_std_ms": float(std_lat),
        "fps": float(fps),
        "per_class": per_class,
    }
    
    output_path = os.path.join(results_dir, f"{model_name}_metrics.json")
    with open(output_path, "w") as f:
        json.dump(result_data, f, indent=2)
    print(f"Results saved to {output_path}")
    
    del model
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return result_data


## Define Mish Activation

In [5]:
import torch.nn as nn
import torch.nn.functional as F

class Mish(nn.Module):
    """Mish activation function: f(x) = x * tanh(softplus(x))"""
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

# Verify
test = torch.randn(1, 64, 32, 32)
mish = Mish()
out = mish(test)
assert out.shape == test.shape
print(f"Mish verified: {test.shape} -> {out.shape}")
print(f"Mish adds 0 trainable parameters (activation only)")


Mish verified: torch.Size([1, 64, 32, 32]) -> torch.Size([1, 64, 32, 32])
Mish adds 0 trainable parameters (activation only)


## Build YOLO11s-P2 + Mish Model

In [6]:
from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn import tasks as tasks_module

MODEL_CFG = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\configs\yolo11s-p2.yaml"
MODEL_NAME = "mish"

# Register Mish with Ultralytics
tasks_module.Mish = Mish

model = YOLO(MODEL_CFG)

# Replace SiLU with Mish in NECK layers only
# Head/Neck starts at layer 11 (after backbone layer 10 = C2PSA)
# We keep backbone activations as SiLU for pretrained compatibility

silu_count = 0
mish_count = 0
mish_locations = []

# Count backbone SiLU (layers 0-10)
backbone_silu = 0
for i in range(11):  # backbone layers 0-10
    for name, mod in model.model.model[i].named_modules():
        if isinstance(mod, nn.SiLU):
            backbone_silu += 1

print(f"Backbone SiLU activations (kept): {backbone_silu}")

# Replace SiLU in neck/head layers (11-28)
for i in range(11, len(model.model.model) - 1):  # Exclude Detect layer
    layer = model.model.model[i]
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.SiLU):
            silu_count += 1
    
    # Replace SiLU with Mish in Conv blocks within neck
    def replace_silu_with_mish(module, layer_idx):
        count = 0
        for name, child in module.named_children():
            if isinstance(child, nn.SiLU):
                setattr(module, name, Mish())
                count += 1
            else:
                count += replace_silu_with_mish(child, layer_idx)
        return count
    
    replaced = replace_silu_with_mish(layer, i)
    if replaced > 0:
        mish_count += replaced
        mish_locations.append(f"layer {i}: {replaced} Mish activations")

print(f"\nNeck/Head SiLU replaced with Mish: {mish_count}")
print(f"Mish locations:")
for loc in mish_locations:
    print(f"  {loc}")

# Final count
total_silu = 0
total_mish = 0
for name, mod in model.model.named_modules():
    if isinstance(mod, nn.SiLU):
        total_silu += 1
    elif isinstance(mod, Mish):
        total_mish += 1

print(f"\nFinal activation counts:")
print(f"  SiLU (remaining, backbone): {total_silu}")
print(f"  Mish (new, neck/head): {total_mish}")

assert total_mish > 0, "FAILED: No Mish activations found!"
print("\n[OK] Mish activation replacement verified")


Backbone SiLU activations (kept): 11

Neck/Head SiLU replaced with Mish: 32
Mish locations:
  layer 13: 4 Mish activations
  layer 16: 4 Mish activations
  layer 19: 4 Mish activations
  layer 20: 1 Mish activations
  layer 22: 4 Mish activations
  layer 23: 1 Mish activations
  layer 25: 4 Mish activations
  layer 26: 1 Mish activations
  layer 28: 9 Mish activations

Final activation counts:
  SiLU (remaining, backbone): 1
  Mish (new, neck/head): 32

[OK] Mish activation replacement verified


## Verify Mish Model

In [7]:
# Model parameters (should be same as baseline since Mish has no params)
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Note: Mish does not add trainable parameters")

# Forward pass
x = torch.randn(1, 3, 640, 640)
model.model.eval()
with torch.no_grad():
    output = model.model(x)
print(f"\nForward pass OK")

# Verify Mish is actually in computation path
model.model.train()
x = torch.randn(1, 3, 640, 640, requires_grad=True)
output = model.model(x)
if isinstance(output, dict):
    loss = sum(v.sum() for v in output.values() if isinstance(v, torch.Tensor))
elif isinstance(output, (list, tuple)):
    loss = sum(o.sum() for o in output if isinstance(o, torch.Tensor))
else:
    loss = output.sum()
loss.backward()
print("Gradient flow through Mish OK")
model.model.zero_grad()


Total parameters: 9,625,968
Trainable parameters: 9,625,952
Note: Mish does not add trainable parameters



Forward pass OK


Gradient flow through Mish OK


## Train Mish Model

In [8]:
# Train
project_dir = os.path.join(r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning", "runs", "mish")
train_config = TRAIN_CONFIG.copy()
train_config["project"] = project_dir
train_config["name"] = "train"
train_config["exist_ok"] = True

print(f"Starting MISH model training...")

import os
best_path_check = os.path.join(project_dir, 'train', 'weights', 'best.pt')
if os.path.exists(best_path_check):
    print(f"Found {best_path_check}, skipping training!")
    results = None
else:
    results = model.train(**train_config)

print("\nTraining complete!")


Starting MISH model training...
Found D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\runs\mish\train\weights\best.pt, skipping training!

Training complete!


## Evaluate and Benchmark

In [9]:
# Evaluate best checkpoint
best_path = os.path.join(project_dir, "train", "weights", "best.pt")
if not os.path.exists(best_path):
    import glob
    best_candidates = glob.glob(os.path.join(project_dir, "**/best.pt"), recursive=True)
    if best_candidates:
        best_path = best_candidates[0]

print(f"Best checkpoint: {best_path}")

# Re-register Mish for loading
tasks_module.Mish = Mish

model = YOLO(best_path)
val_results = model.val(data=TRAIN_CONFIG["data"], imgsz=TRAIN_CONFIG["imgsz"], device=DEVICE, workers=0)

print("\nValidation Results:")
print(f"  Precision: {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"  Recall: {val_results.results_dict['metrics/recall(B)']:.4f}")
print(f"  mAP50: {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP50-95: {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")

benchmark_results = benchmark_model(best_path, DEVICE)
result_data = save_experiment_results(MODEL_NAME, val_results, best_path, benchmark_results, RESULTS_DIR)

print("\n" + "=" * 60)
print("MISH RESULTS SUMMARY")
print("=" * 60)
for k, v in result_data.items():
    if k not in ["model_path", "per_class"]:
        print(f"  {k}: {v}")

del model, val_results
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nDone!")


Best checkpoint: D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\runs\mish\train\weights\best.pt


Ultralytics 8.4.92  Python-3.13.6 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)


YOLO11s-p2 summary (fused): 121 layers, 9,559,900 parameters, 0 gradients, 28.6 GFLOPs


val: Fast image access  (ping: 0.10.0 ms, read: 668.6137.6 MB/s, size: 102.6 KB)


val: Scanning D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\dataset\labels\val.cache... 401 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 401/401 98.9Mit/s 0.0s

...

WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.764739990234375. Dividing input by 255.


Latency: 21.39 +/- 5.27 ms


FPS: 46.7


YOLO11s-p2 summary: 217 layers, 9,575,292 parameters, 0 gradients, 29.0 GFLOPs


Results saved to D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results\mish_metrics.json

MISH RESULTS SUMMARY
  model_name: mish
  n_layers: 217
  n_params: 9575292
  n_grads: 0
  gflops: 28.953856000000002
  model_size_mb: 18.680057525634766
  precision: 0.4488856461236846
  recall: 0.4142945229029122
  mAP50: 0.38802786118026145
  mAP50-95: 0.16983034070067363
  latency_ms: 21.390594999829773
  latency_std_ms: 5.272917450286252
  fps: 46.74951772065985

Done!
